# Movie Expert Agent (#2.2 — 프레임워크 없이)

OpenAI API + 실제 영화 API로 만드는 첫 에이전트. **TODO를 직접 채워라.** 막히면 채팅 가이드 참고.

- API base: `https://nomad-movies.nomadcoders.workers.dev`
- 모델: `gpt-4o-mini`
- 함수 3개: `get_popular_movies()` · `get_movie_details(id)` · `get_movie_credits(id)`

흐름: 프롬프트로 함수 설명 → 모델이 `{함수명,인자}` JSON 출력 → **내 코드가 실행** → 결과를 다시 모델에 → 자연어 답변

## 0. 준비 — import & 클라이언트

In [36]:
# TODO: 필요한 모듈 import (os, json, requests, openai, dotenv 의 load_dotenv)
# TODO: load_dotenv() 로 .env 의 OPENAI_API_KEY 로딩
# TODO: 키가 잡혔는지 bool 로만 확인 (값 전체는 찍지 말 것)
import os, json, requests, openai
from dotenv import load_dotenv

load_dotenv()
key = os.getenv("OPENAI_API_KEY")
print("key loaded:", bool(key))      

key loaded: True


In [37]:
# TODO: client = ?   (인자 없이 생성하면 OPENAI_API_KEY 를 자동으로 읽는다)
client = openai.OpenAI(api_key=key)

## 1. 에이전트가 호출할 실제 함수 3개
`requests.get(...).json()` 으로 각 엔드포인트를 친다. 모델은 이 함수를 직접 못 돌린다 — 이름만 고르고 실행은 내 코드가.

In [38]:
BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

# TODO: get_popular_movies()        -> GET /movies
def get_popular_movies():
    return requests.get(f"{BASE_URL}/movies").json()

# TODO: get_movie_details(movie_id) -> GET /movies/{movie_id}
def get_movie_details(movie_id):
    return requests.get(f"{BASE_URL}/movies/{movie_id}").json()

# TODO: get_movie_credits(movie_id) -> GET /movies/{movie_id}/credits
def get_movie_credits(movie_id):
    return requests.get(f"{BASE_URL}/movies/{movie_id}/credits").json()


In [39]:
# TODO: FUNCTION_MAP = { "함수이름문자열": 실제함수객체, ... }
#       모델이 돌려준 이름 문자열을 실제 함수로 잇는 디스패처
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,   # key=문자열, value=함수객체(괄호 X!)
    "get_movie_details":get_movie_details,
    "get_movie_credits":get_movie_credits
}

## 2. 함수를 설명하는 프롬프트
모델에게 함수 3개와 인자를 글로 설명하고 **JSON 한 줄로만** 답하라고 강제. 예시 형태도 프롬프트에 박아라:
`{"function": "get_movie_details", "arguments": {"movie_id": 550}}`

In [40]:
# TODO: SYSTEM_PROMPT = """..."""
#   - 함수 3개 이름/인자/설명
#   - "JSON 객체만 출력, 다른 텍스트·코드펜스 금지"
#   - 인자 없을 땐 "arguments": {}
# SYSTEM_PROMPT = """내 시스템에는 다음 함수가 있다:
# - get_popular_movies
# - get_movie_details
# - get_movie_credits

# 함수는 영화의 id 값을 인자로 받는다. 예: get_movie_details(movie_id)
# 인자 없을 땐 "arguments": {}

# "JSON 객체만 출력, 다른 텍스트·코드펜스 금지"""

# 지금: "JSON 객체만 출력해"라고만 했지, 그 JSON이 어떤 모양이어야 하는지를 안 알려줬어. 
# 모델은 형식을 모르면 제멋대로 답해. 그리고 나중에 ask_which_function이 이렇게 꺼낼 거야:
# call = json.loads(raw)
# call["function"]    # ← "function" 키
# call["arguments"]   # ← "arguments" 키
# → 즉 모델이 정확히 {"function": ..., "arguments": ...} 로 답하게 프롬프트에 그 모양을 박아야 해.

# 고칠 포인트 2개
# 출력 형식(shape)을 예시로 명시 — 이런 식으로:


# {"function": "함수명", "arguments": {인자}}
# 인자 있는 예: {"function": "get_movie_details", "arguments": {"movie_id": 550}}
# 인자 없는 예: {"function": "get_popular_movies", "arguments": {}}
# 어떤 함수가 인자를 받는지 구분 — 지금은 "함수는 id를 받는다"고 뭉뚱그렸는데:

# get_popular_movies → 인자 없음
# get_movie_details(movie_id), get_movie_credits(movie_id) → movie_id 받음
# 즉 프롬프트에 들어가야 할 뼈대
# 함수 3개 + 각자 인자 설명 (popular는 인자 없음 명시)
# "반드시 이 JSON 형태로만 답해: {"function": ..., "arguments": ...}"
# 위 두 예시(인자 있을 때 / 없을 때)

SYSTEM_PROMPT = """너는 영화 전문가 에이전트다. 사용할 수 있는 함수는 다음 3개뿐이다:

- get_popular_movies: 인기 영화 목록을 가져온다. 인자 없음.
- get_movie_details: 특정 영화의 상세 정보를 가져온다. 인자: movie_id (정수).
- get_movie_credits: 특정 영화의 출연진/제작진을 가져온다. 인자: movie_id (정수).

사용자의 질문을 보고 어떤 함수를 어떤 인자로 호출할지 정해라.
반드시 아래 JSON 형태로만 답하라. 다른 설명이나 코드펜스는 절대 쓰지 마라.

{"function": "함수이름", "arguments": {인자들}}

예시:
- 인기 영화 → {"function": "get_popular_movies", "arguments": {}}
- 550 영화 정보 → {"function": "get_movie_details", "arguments": {"movie_id": 550}}
- 550 출연진 → {"function": "get_movie_credits", "arguments": {"movie_id": 550}}
"""


## 3. 1단계 — 모델이 함수명+인자를 맞게 출력하는지 확인
챌린지 핵심 요구. 모델 호출 → `choices[0].message.content` 의 JSON 문자열 → `json.loads` 로 dict.

In [41]:
# def ask_which_function(user_input):
#     # TODO: client.chat.completions.create(model="gpt-4o-mini", messages=[system, user])
#     # TODO: raw = response.choices[0].message.content  (출력해서 눈으로 확인)
#     # TODO: json.loads 로 dict 반환 (모델이 잡텍스트 섞을 수 있으니 첫 '{' ~ 마지막 '}' 만 잘라도 좋음)
#     pass

def ask_which_function(user_input):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_input},
        ],
    )
    raw = response.choices[0].message.content
    print("raw:", raw)                      # 모델이 뭘 뱉었는지 눈으로 확인
    start, end = raw.find("{"), raw.rfind("}")
    return json.loads(raw[start:end + 1])   # 첫 { ~ 마지막 } 만 잘라 dict로



In [42]:
# 세 입력의 함수 선택이 맞는지 먼저 점검 (run_agent 만들기 전 검증)
for q in [
    "지금 인기 있는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘",
]:
    print(q, "->", ask_which_function(q))


raw: {"function": "get_popular_movies", "arguments": {}}
지금 인기 있는 영화가 무엇인지 알려줘 -> {'function': 'get_popular_movies', 'arguments': {}}
raw: {"function": "get_movie_details", "arguments": {"movie_id": 550}}
movie ID 550에 해당하는 영화가 무엇인지 알려줘 -> {'function': 'get_movie_details', 'arguments': {'movie_id': 550}}
raw: {"function": "get_movie_credits", "arguments": {"movie_id": 550}}
movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘 -> {'function': 'get_movie_credits', 'arguments': {'movie_id': 550}}


## 4. 2단계 — 함수 실행 + 결과로 자연어 답변 (풀 에이전트)
모델이 고른 함수를 `FUNCTION_MAP[name](**arguments)` 로 실행 → 결과를 다시 모델에 넣어 한국어 답변.

In [43]:
def run_agent(user_input):
    # 1) 어떤 함수 부를지 (방금 만든 함수 재사용)
    call = ask_which_function(user_input)
    name = call["function"]
    args = call.get("arguments") or {}

    # 2) 그 함수를 실제로 실행
    result = FUNCTION_MAP[name](**args)

    # 3) 실행 결과를 모델에 주고, 자연어 답변 받기
    result_text = json.dumps(result, ensure_ascii=False)[:4000]   # 너무 크면 잘라서 (토큰 절약)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "너는 영화 전문가다. 주어진 데이터만 근거로 한국어로 자연스럽게 답해라."},
            {"role": "user", "content": user_input},
            {"role": "assistant", "content": f"{name}({args}) 호출 결과: {result_text}"},
            {"role": "user", "content": "위 데이터로 내 질문에 답해줘."},
        ],
    )
    return response.choices[0].message.content


## 5. 테스트 — 챌린지 지정 입력 3개

In [ ]:
run_agent("지금 인기 있는 영화가 무엇인지 알려줘")

raw: {"function": "get_popular_movies", "arguments": {}}


In [ ]:
run_agent("movie ID 550에 해당하는 영화가 무엇인지 알려줘")

raw: {"function": "get_movie_details", "arguments": {"movie_id": 550}}


'영화 ID 550에 해당하는 영화는 **"Fight Club"**입니다. 이 영화는 1999년 10월 15일에 개봉하였으며, 장르는 드라마와 스릴러입니다. 감독은 데이비드 핀처이며, 주요 줄거리는 불면증에 시달리는 남성과 비누 판매사가 남성의 본능적인 공격성을 새로운 치료법으로 발휘하는 이야기입니다. 영화는 평점 8.4를 받았으며, 평범한 삶에서 벗어나기 위한 Underground "파이트 클럽" 형성을 중심으로 전개됩니다. \n\n영화의 포스터는 아래와 같습니다:\n![Fight Club](https://image.tmdb.org/t/p/w780/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg)\n\n자세한 정보는 [여기](https://www.20thcenturystudios.com/movies/fight-club)에서 확인할 수 있습니다.'

In [ ]:
run_agent("movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘")

raw: {"function": "get_movie_credits", "arguments": {"movie_id": 550}}


'영화 ID 550에 해당하는 영화는 "파이팅"이며, 주요 출연진은 다음과 같습니다:\n\n1. 에드워드 노턴 (Edward Norton) - 캐릭터: 내레이터 (Narrator)\n   ![에드워드 노턴](https://image.tmdb.org/t/p/w185/8nytsqL59SFJTVYVrN72k6qkGgJ.jpg)\n   \n2. 브래드 피트 (Brad Pitt) - 캐릭터: 타일러 더든 (Tyler Durden)\n   ![브래드 피트](https://image.tmdb.org/t/p/w185/m09Y1YfPPeNYYUSHnnVqahkrC1o.jpg)\n   \n3. 헬레나 본햄 카터 (Helena Bonham Carter) - 캐릭터: 말라 싱어 (Marla Singer)\n   ![헬레나 본햄 카터](https://image.tmdb.org/t/p/w185/hJMbNSPJ2PCahsP3rNEU39C8GWU.jpg)\n   \n4. 미트 로프 (Meat Loaf) - 캐릭터: 로버트 폴슨 (Robert Paulson)\n   ![미트 로프](https://image.tmdb.org/t/p/w185/1zkohpaG3my4qQAZGVgzgPuXwZ6.jpg)\n\n5. 자레드 레토 (Jared Leto) - 캐릭터: 앤젤 페이스 (Angel Face)\n   ![자레드 레토](https://image.tmdb.org/t/p/w185/ca3x0OfIKbJppZh8S1Alx3GfUZO.jpg)\n\n이 외에도 여러 배우들이 출연하고 있습니다. 영화의 매력적인 캐릭터들과 그들의 연기를 통해 많은 사랑을 받았습니다.'